# Master Script — CLASSIFICAÇÃO (scikit-learn)

**Guia de consulta.** Não foi feito para rodar de ponta a ponta: é um repositório de
sintaxe, pipelines e grades de hiperparâmetros para copiar quando um projeto novo começa.

### Fluxo do documento
| # | Etapa | O que tem aqui |
|---|-------|----------------|
| 0 | Setup | Imports canônicos, `RANDOM_STATE`, config global |
| 1 | Data Prep | Leitura, tipagem, duplicatas, missing, `X` / `y` |
| 2 | Split | `train_test_split` estratificado, quando usar validação separada |
| 3 | Pipelines | `ColumnTransformer` + um pipeline por família matemática |
| 4 | Validação Cruzada | `cross_validate` comparando todas as famílias |
| 5 | Otimização | `RandomizedSearchCV` + **grades de todas as famílias** |
| 6 | Prova Final | Métricas, matriz de confusão, ROC/PR, threshold, persistência |
| A | Apêndice | Cheatsheet de escolha de família e de métrica |

### Regra de ouro
Tudo que **aprende** com os dados (imputação, escala, encoding, seleção de variável,
balanceamento) mora **dentro do Pipeline**. Fora do Pipeline = vazamento de dados
(*data leakage*) e métrica de validação inflada.

---
## 0. Setup e imports

In [1]:
# --- Núcleo ---------------------------------------------------------------
import numpy as np
import pandas as pd

# --- Pré-processamento ----------------------------------------------------
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler,      # média 0, desvio 1  -> padrão para modelos com distância/gradiente
    RobustScaler,        # usa mediana/IQR    -> quando há outliers
    MinMaxScaler,        # [0, 1]             -> redes neurais, dados esparsos positivos
    PowerTransformer,    # yeo-johnson        -> aproxima da normal (assimetria forte)
    QuantileTransformer, # rank -> normal/uniforme (robusto e agressivo)
    OneHotEncoder,       # categóricas para modelos lineares/distância
    OrdinalEncoder,      # categóricas para modelos baseados em árvore
    PolynomialFeatures,
)
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE

# --- Split e validação ----------------------------------------------------
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold, RepeatedStratifiedKFold,
    StratifiedGroupKFold,      # quando há grupos (ex.: mesmo cliente em várias linhas)
    TimeSeriesSplit,           # quando há ordem temporal
    cross_validate, cross_val_predict,
    GridSearchCV, RandomizedSearchCV,
)

# --- Modelos: famílias matemáticas ----------------------------------------
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB, ComplementNB
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier,
    BaggingClassifier, VotingClassifier, StackingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier          # baseline obrigatório

# --- Métricas -------------------------------------------------------------
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    matthews_corrcoef, cohen_kappa_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    precision_recall_curve, roc_curve, make_scorer,
)
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt
import joblib

# --- Config global --------------------------------------------------------
RANDOM_STATE = 42
N_JOBS = -1                     # usa todos os núcleos
pd.set_option("display.max_columns", 100)
np.random.seed(RANDOM_STATE)

In [2]:
# --- Busca sucessiva (experimental): 3x a 10x mais rápida que a busca comum ---
# Avalia muitos candidatos com poucos dados e vai eliminando os piores.
from sklearn.experimental import enable_halving_search_cv  # noqa: F401  (import obrigatório)
from sklearn.model_selection import HalvingRandomSearchCV, HalvingGridSearchCV

# --- Bibliotecas externas (mesma API do sklearn, entram no Pipeline igual) ---
# pip install xgboost lightgbm catboost imbalanced-learn
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# imbalanced-learn: Pipeline próprio, porque reamostragem só pode agir no TREINO de cada fold
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, SMOTENC, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek

---
## 1. Carregamento e Limpeza (Data Prep)

Esqueleto do preparo padrão. Os dados abaixo são fictícios e servem só para dar forma ao
resto do documento: numéricas + categóricas + faltantes + alvo binário desbalanceado.

In [3]:
# ==== 1.1 Leitura ==========================================================
# df = pd.read_csv("dados.csv", sep=",", decimal=".", encoding="utf-8")
# df = pd.read_parquet("dados.parquet")            # preferível: preserva dtypes
# df = pd.read_excel("dados.xlsx", sheet_name=0)
# df = pd.read_sql("SELECT * FROM tabela", con=engine)

# ---- DataFrame fictício (substituir pela leitura real) --------------------
n = 1_000
rng = np.random.default_rng(RANDOM_STATE)

df = pd.DataFrame({
    "idade":        rng.integers(18, 80, n).astype(float),
    "renda":        rng.lognormal(9, 0.6, n),           # cauda longa -> candidata a log/Power
    "score":        rng.normal(600, 90, n),
    "tempo_casa":   rng.exponential(4, n),
    "n_produtos":   rng.integers(1, 6, n).astype(float),
    "regiao":       rng.choice(["Norte", "Sul", "Sudeste", "Nordeste"], n),
    "canal":        rng.choice(["app", "web", "loja"], n, p=[.5, .3, .2]),
    "plano":        rng.choice(["basico", "plus", "premium"], n),   # categórica ORDINAL
    "id_cliente":   np.arange(n),                        # identificador -> descartar
    "churn":        rng.choice([0, 1], n, p=[.85, .15]), # alvo desbalanceado (15%)
})

# Injeta faltantes de propósito, para o pipeline ter o que imputar
for col, frac in [("renda", .08), ("score", .05), ("canal", .03)]:
    df.loc[df.sample(frac=frac, random_state=RANDOM_STATE).index, col] = np.nan

df.head()

,idade,renda,score,tempo_casa,n_produtos,regiao,canal,plano,id_cliente,churn
0,23.0,19147.524244,650.698177,3.324975,3.0,Sul,app,plus,0,0
1,65.0,8560.484833,610.312599,8.682105,2.0,Sudeste,app,premium,1,0
2,58.0,11481.200725,629.694918,3.781799,4.0,Norte,app,basico,2,1
3,45.0,7831.662441,669.984858,0.032991,4.0,Norte,web,premium,3,0
4,44.0,7315.535023,527.146231,1.539684,1.0,Nordeste,web,premium,4,0


In [5]:
# ==== 1.2 Diagnóstico rápido ==============================================
print(df.shape)
df.info()
df.describe(include="all").T

# Faltantes por coluna
faltantes = df.isna().mean().sort_values(ascending=False)
print(faltantes[faltantes > 0].mul(100).round(2).astype(str) + " %")

# Balanceamento do alvo -> decide métrica, class_weight e estratificação
print(df["churn"].value_counts(normalize=True))

(1000, 10)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   idade       1000 non-null   float64
 1   renda       920 non-null    float64
 2   score       950 non-null    float64
 3   tempo_casa  1000 non-null   float64
 4   n_produtos  1000 non-null   float64
 5   regiao      1000 non-null   str    
 6   canal       970 non-null    str    
 7   plano       1000 non-null   str    
 8   id_cliente  1000 non-null   int64  
 9   churn       1000 non-null   int64  
dtypes: float64(5), int64(2), str(3)
memory usage: 92.7 KB
renda    8.0 %
score    5.0 %
canal    3.0 %
dtype: str
churn
0    0.862
1    0.138
Name: proportion, dtype: float64


In [6]:
# ==== 1.3 Limpeza padrão ==================================================
TARGET = "churn"

# a) Descartar identificadores e vazamentos (colunas que só existem depois do evento)
DROP = ["id_cliente"]
df = df.drop(columns=DROP, errors="ignore")

# b) Duplicatas
df = df.drop_duplicates()

# c) Tipagem explícita (evita que categórica numerada vire número)
CAT_COLS = ["regiao", "canal", "plano"]
NUM_COLS = ["idade", "renda", "score", "tempo_casa", "n_produtos"]
df[CAT_COLS] = df[CAT_COLS].astype("category")
df[NUM_COLS] = df[NUM_COLS].apply(pd.to_numeric, errors="coerce")

# d) Linhas sem alvo são inúteis em aprendizado supervisionado -> remover
df = df.dropna(subset=[TARGET])

# NOTA: imputação, escala e encoding NÃO entram aqui. Vão no Pipeline (etapa 3),
# para serem ajustados só no treino de cada fold.

# e) Separação X / y
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

# Se o alvo for texto: LabelEncoder guarda a ordem das classes para os relatórios
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder(); y = le.fit_transform(df[TARGET]); CLASSES = le.classes_

CLASSES = np.array(["nao_churn", "churn"])
print(X.shape, y.shape, y.mean().round(3))

(1000, 8) (1000,) 0.138


---
## 2. Separação dos Dados (Split)

**Estratificado sempre** em classificação — mantém a proporção das classes nos dois lados.
O teste é lacrado: só é aberto na etapa 6.

| Situação | Split |
|---|---|
| Tabular comum | `train_test_split(..., stratify=y)` |
| Linhas repetidas por entidade (cliente, paciente) | `StratifiedGroupKFold` / `GroupShuffleSplit` |
| Série temporal | corte por data + `TimeSeriesSplit` (**nunca** embaralhar) |
| Deep learning / early stopping / calibração | treino + **validação** + teste (3 partes) |

In [7]:
# ==== 2.1 Split padrão: 80 / 20 estratificado =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,              # obrigatório em classificação
    random_state=RANDOM_STATE,
    shuffle=True,
)
print(f"treino: {X_train.shape} | teste: {X_test.shape}")
print(y_train.value_counts(normalize=True).round(3).to_dict())
print(y_test.value_counts(normalize=True).round(3).to_dict())

treino: (800, 8) | teste: (200, 8)
{0: 0.862, 1: 0.138}
{0: 0.86, 1: 0.14}


In [8]:
# ==== 2.2 Split de 3 vias: quando existe VALIDAÇÃO separada ===============
# Usar quando o modelo precisa de early stopping (XGB/LGBM/MLP), calibração de
# probabilidade ou ajuste de threshold — coisas que não podem tocar o teste.
# Alvo: 60 / 20 / 20.

X_temp, X_test2, y_temp, y_test2 = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train2, X_vali, y_train2, y_vali = train_test_split(
    X_temp, y_temp,
    test_size=0.25,          # 0.25 de 80% = 20% do total
    stratify=y_temp,
    random_state=RANDOM_STATE,
)
print(f"treino: {len(X_train2)} | validação: {len(X_vali)} | teste: {len(X_test2)}")

# Com validação cruzada bem feita (etapa 4), o conjunto de validação vira opcional:
# a CV já cumpre esse papel dentro do treino.

treino: 600 | validação: 200 | teste: 200


In [9]:
# ==== 2.3 Variantes de split ==============================================

# --- Por grupo: nenhuma entidade aparece nos dois lados --------------------
# from sklearn.model_selection import GroupShuffleSplit
# grupos = df["id_cliente"]
# gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
# idx_tr, idx_te = next(gss.split(X, y, groups=grupos))
# X_train, X_test = X.iloc[idx_tr], X.iloc[idx_te]

# --- Temporal: corte por data, sem embaralhar -----------------------------
# corte = df["data"].quantile(0.8)
# X_train, y_train = X[df["data"] <= corte], y[df["data"] <= corte]
# X_test,  y_test  = X[df["data"] >  corte], y[df["data"] >  corte]

# --- Estratégia de validação cruzada usada da etapa 4 em diante -----------
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Mais estável (e mais caro) quando o dataset é pequeno:
# CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

---
## 3. Construção dos Pipelines

Cada família matemática tem exigências diferentes de pré-processamento. São **três
receitas de `ColumnTransformer`** que cobrem praticamente tudo:

| Receita | Numéricas | Categóricas | Para quem |
|---|---|---|---|
| `"scaled"` | mediana + `StandardScaler` | moda + `OneHotEncoder` | Linear, SVM, KNN, MLP, LDA/QDA |
| `"tree"` | mediana, **sem escala** | moda + `OrdinalEncoder` | Árvore, RF, ET, GB, AdaBoost |
| `"native"` | mediana, sem escala | `OrdinalEncoder` marcada como categórica | HistGB, LightGBM, XGBoost, CatBoost |

Regras que evitam a maioria dos erros:
- **Escala** importa para tudo que usa distância, gradiente ou penalização (`L1`/`L2`). É indiferente para árvores.
- **One-Hot** em árvore com categórica de alta cardinalidade fragmenta os splits e piora o modelo → use `OrdinalEncoder` ou suporte nativo.
- `handle_unknown="ignore"` (OHE) e `handle_unknown="use_encoded_value"` (Ordinal) evitam quebra quando aparece categoria nova no teste.

In [10]:
# ==== 3.1 Seleção de colunas ==============================================
# Fixo (explícito, recomendado em produção):
NUM_FEATURES = ["idade", "renda", "score", "tempo_casa", "n_produtos"]
CAT_FEATURES = ["regiao", "canal", "plano"]

# Dinâmico (útil em exploração — pega as colunas por dtype):
num_selector = make_column_selector(dtype_include=np.number)
cat_selector = make_column_selector(dtype_include=["object", "category"])

In [11]:
# ==== 3.2 Fábrica de pré-processadores ====================================
def build_preprocessor(kind="scaled", scaler=None, num=NUM_FEATURES, cat=CAT_FEATURES):
    # kind: "scaled" | "tree" | "native"
    if kind == "scaled":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler or StandardScaler()),
            # ("power", PowerTransformer(method="yeo-johnson")),  # assimetria forte
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent", fill_value="AUSENTE")),
            ("encoder", OneHotEncoder(
                handle_unknown="infrequent_if_exist",  # categoria nova cai em "infrequent"
                min_frequency=0.01,                    # agrupa categorias raras
                sparse_output=False,
                drop=None,          # use drop="first" só em modelo linear sem regularização
            )),
        ])

    elif kind == "tree":
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1,
                encoded_missing_value=-2,
            )),
        ])

    elif kind == "native":   # HistGB / LGBM / XGB tratam a categórica sozinhos
        num_pipe = Pipeline([("passthrough", "passthrough")])
        cat_pipe = Pipeline([
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1,
                encoded_missing_value=np.nan,
            )),
        ])
    else:
        raise ValueError(kind)

    return ColumnTransformer(
        transformers=[("num", num_pipe, num), ("cat", cat_pipe, cat)],
        remainder="drop",              # "passthrough" mantém as colunas não listadas
        verbose_feature_names_out=False,
        n_jobs=None,
    ).set_output(transform="pandas")   # saída em DataFrame -> nomes preservados

In [12]:
# ==== 3.3 Um Pipeline por família matemática ==============================
# Cada valor é um Pipeline completo: pré-processamento + estimador.
# O passo final chama-se sempre "model" -> as grades usam o prefixo "model__".

PIPELINES = {

    # ---------------- Baseline (referência mínima) ------------------------
    "Dummy": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
    ]),

    # ---------------- Lineares / probabilísticos --------------------------
    # Fronteira de decisão linear. Rápidos, interpretáveis, exigem escala.
    "LogisticRegression": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", LogisticRegression(
            max_iter=2000, class_weight="balanced",
            solver="lbfgs", random_state=RANDOM_STATE)),
    ]),
    "RidgeClassifier": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", RidgeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "SGDClassifier": Pipeline([          # escala para milhões de linhas
        ("prep", build_preprocessor("scaled")),
        ("model", SGDClassifier(
            loss="log_loss", early_stopping=True,
            class_weight="balanced", random_state=RANDOM_STATE)),
    ]),

    # ---------------- Margem máxima (kernel) ------------------------------
    # Excelente em espaço de alta dimensão; custo ~O(n²) -> evitar acima de ~50k linhas.
    "SVC_rbf": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", SVC(kernel="rbf", probability=True,
                      class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "LinearSVC": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", LinearSVC(class_weight="balanced", dual="auto",
                            max_iter=5000, random_state=RANDOM_STATE)),
    ]),

    # ---------------- Baseado em instância (distância) --------------------
    # Sem treino real; sofre com muitas dimensões (maldição da dimensionalidade).
    "KNN": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", KNeighborsClassifier(n_jobs=N_JOBS)),
    ]),

    # ---------------- Bayesiano ------------------------------------------
    # Assume independência entre variáveis. Rapidíssimo, ótimo baseline em texto.
    "GaussianNB": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", GaussianNB()),
    ]),

    # ---------------- Discriminante (redução + classificação) -------------
    "LDA": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", LinearDiscriminantAnalysis()),
    ]),
    "QDA": Pipeline([
        ("prep", build_preprocessor("scaled")),
        ("model", QuadraticDiscriminantAnalysis()),
    ]),

    # ---------------- Árvore isolada --------------------------------------
    # Interpretável, captura não-linearidade, mas sozinha tem variância alta.
    "DecisionTree": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_STATE)),
    ]),

    # ---------------- Ensembles: Bagging (reduz variância) ----------------
    "RandomForest": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", RandomForestClassifier(
            n_estimators=500, class_weight="balanced_subsample",
            n_jobs=N_JOBS, random_state=RANDOM_STATE)),
    ]),
    "ExtraTrees": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", ExtraTreesClassifier(
            n_estimators=500, class_weight="balanced",
            n_jobs=N_JOBS, random_state=RANDOM_STATE)),
    ]),

    # ---------------- Ensembles: Boosting (reduz viés) --------------------
    # Estado da arte em dados tabulares.
    "GradientBoosting": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
    "HistGradientBoosting": Pipeline([   # o boosting rápido nativo do sklearn
        ("prep", build_preprocessor("native")),
        ("model", HistGradientBoostingClassifier(
            categorical_features=[len(NUM_FEATURES) + i for i in range(len(CAT_FEATURES))],
            class_weight="balanced",
            early_stopping=True, validation_fraction=0.1,
            random_state=RANDOM_STATE)),
    ]),
    "AdaBoost": Pipeline([
        ("prep", build_preprocessor("tree")),
        ("model", AdaBoostClassifier(random_state=RANDOM_STATE)),
    ]),

    # ---------------- Rede neural ----------------------------------------
    # Exige escala e bastante dado; raramente vence boosting em tabular.
    "MLP": Pipeline([
        ("prep", build_preprocessor("scaled", scaler=MinMaxScaler())),
        ("model", MLPClassifier(
            max_iter=500, early_stopping=True, random_state=RANDOM_STATE)),
    ]),

    # ---------------- Boosting externo ------------------------------------
    "XGBoost": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", XGBClassifier(
            n_estimators=600, learning_rate=0.05,
            tree_method="hist", enable_categorical=False,
            eval_metric="logloss",
            scale_pos_weight=float((y_train == 0).sum() / (y_train == 1).sum()),
            n_jobs=N_JOBS, random_state=RANDOM_STATE)),
    ]),
    "LightGBM": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", LGBMClassifier(
            n_estimators=600, learning_rate=0.05,
            class_weight="balanced", verbose=-1,
            n_jobs=N_JOBS, random_state=RANDOM_STATE)),
    ]),
    "CatBoost": Pipeline([
        ("prep", build_preprocessor("native")),
        ("model", CatBoostClassifier(
            iterations=600, learning_rate=0.05, depth=6,
            auto_class_weights="Balanced",
            verbose=0, random_seed=RANDOM_STATE)),
    ]),
}

print(f"{len(PIPELINES)} pipelines declarados")

20 pipelines declarados


In [13]:
# ==== 3.4 Blocos opcionais dentro do Pipeline =============================

# --- Redução de dimensionalidade (muitas colunas / colinearidade) ---------
pipe_pca = Pipeline([
    ("prep", build_preprocessor("scaled")),
    ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),  # 95% da variância
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

# --- Seleção de variáveis ------------------------------------------------
pipe_select = Pipeline([
    ("prep", build_preprocessor("tree")),
    ("select", SelectKBest(score_func=mutual_info_classif, k=10)),
    ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
])

# --- Interações polinomiais (dar não-linearidade a modelo linear) --------
pipe_poly = Pipeline([
    ("prep", build_preprocessor("scaled")),
    ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ("model", LogisticRegression(max_iter=5000, penalty="l2", random_state=RANDOM_STATE)),
])

# --- Ensembles de segundo nível ------------------------------------------
voting = VotingClassifier(
    estimators=[
        ("lr", PIPELINES["LogisticRegression"]),
        ("rf", PIPELINES["RandomForest"]),
        ("gb", PIPELINES["HistGradientBoosting"]),
    ],
    voting="soft",            # média das probabilidades (exige predict_proba)
    weights=[1, 2, 2],
    n_jobs=N_JOBS,
)

stacking = StackingClassifier(
    estimators=[
        ("rf", PIPELINES["RandomForest"]),
        ("gb", PIPELINES["HistGradientBoosting"]),
        ("svc", PIPELINES["SVC_rbf"]),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5,                     # gera as meta-features sem vazamento
    passthrough=False,
    n_jobs=N_JOBS,
)

In [14]:
# ==== 3.5 Desbalanceamento: três estratégias ==============================

# (A) Peso de classe — primeira escolha: não inventa dado, custo zero.
#     class_weight="balanced" em LogisticRegression/SVC/RF/DecisionTree
#     scale_pos_weight = n_neg / n_pos  em XGBoost

# (B) Reamostragem — usa ImbPipeline, para agir só no treino de cada fold.
pipe_smote = ImbPipeline([
    ("prep", build_preprocessor("scaled")),
    ("smote", SMOTE(sampling_strategy=0.5, k_neighbors=5, random_state=RANDOM_STATE)),
    ("under", RandomUnderSampler(sampling_strategy=0.8, random_state=RANDOM_STATE)),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
# SMOTENC quando há categóricas: SMOTENC(categorical_features=[...], random_state=...)

# (C) Mover o threshold — quase sempre o melhor custo-benefício.
#     Treina normal e escolhe o corte na etapa 6 (não é 0.5 por lei).

# ERRO CLÁSSICO: aplicar SMOTE antes do split ou antes da CV.
# Isso replica exemplos sintéticos para a validação e infla a métrica.

---
## 4. Avaliação das Famílias (Validação Cruzada)

Roda **todas as famílias** na mesma CV estratificada, **só com os dados de treino**.
O objetivo aqui não é o número final — é decidir *qual família* levar para a otimização.

### Qual métrica escolher
| Cenário | Métrica |
|---|---|
| Classes equilibradas, erros de custo igual | `accuracy` |
| Classes desbalanceadas | `balanced_accuracy`, `f1`, `matthews_corrcoef` |
| Ranqueamento / escolha de threshold depois | `roc_auc` |
| Desbalanceamento forte (< 5% de positivos) | `average_precision` (AUC-PR) |
| Probabilidade precisa (preço, risco) | `neg_log_loss`, `neg_brier_score` |
| Custo assimétrico (falso negativo caro) | `recall`, `fbeta` com `beta=2` |

Olhe sempre **média ± desvio**. Diferença menor que o desvio entre folds não é diferença.

In [15]:
# ==== 4.1 Métricas avaliadas em cada fold =================================
SCORING = {
    "accuracy":       "accuracy",
    "balanced_acc":   "balanced_accuracy",
    "precision":      "precision",
    "recall":         "recall",
    "f1":             "f1",
    "roc_auc":        "roc_auc",
    "avg_precision":  "average_precision",
    "mcc":            make_scorer(matthews_corrcoef),
    # "f2": make_scorer(fbeta_score, beta=2),   # prioriza recall
}
METRICA_ALVO = "roc_auc"     # critério de ordenação e de refit na etapa 5

In [16]:
# ==== 4.2 Torneio entre as famílias =======================================
import warnings, time
warnings.filterwarnings("ignore")

resultados = []
for nome, pipe in PIPELINES.items():
    t0 = time.perf_counter()
    try:
        cv_res = cross_validate(
            pipe, X_train, y_train,
            cv=CV,
            scoring=SCORING,
            n_jobs=N_JOBS,
            return_train_score=True,   # treino >> teste = overfitting
            error_score="raise",       # "raise" para ver o erro; np.nan para ignorar
        )
        linha = {"modelo": nome}
        for m in SCORING:
            linha[f"{m}_mean"] = cv_res[f"test_{m}"].mean()
            linha[f"{m}_std"]  = cv_res[f"test_{m}"].std()
        linha["train_alvo"] = cv_res[f"train_{METRICA_ALVO}"].mean()
        linha["gap_overfit"] = linha["train_alvo"] - linha[f"{METRICA_ALVO}_mean"]
        linha["fit_time_s"] = cv_res["fit_time"].mean()
        resultados.append(linha)
        print(f"{nome:<22} {METRICA_ALVO}={linha[f'{METRICA_ALVO}_mean']:.4f} "
              f"(+/-{linha[f'{METRICA_ALVO}_std']:.4f})  {time.perf_counter()-t0:.1f}s")
    except Exception as e:
        print(f"{nome:<22} FALHOU: {type(e).__name__}: {e}")

ranking = (pd.DataFrame(resultados)
             .sort_values(f"{METRICA_ALVO}_mean", ascending=False)
             .reset_index(drop=True))
display(ranking.round(4))

Dummy                  roc_auc=0.5273 (+/-0.0387)  2.0s
LogisticRegression     roc_auc=0.5042 (+/-0.0274)  1.3s
RidgeClassifier        roc_auc=0.5035 (+/-0.0272)  0.2s
SGDClassifier          roc_auc=0.4857 (+/-0.0430)  0.2s
SVC_rbf                roc_auc=0.5586 (+/-0.0480)  0.3s
LinearSVC              roc_auc=0.5030 (+/-0.0271)  0.2s
KNN                    roc_auc=0.5293 (+/-0.0827)  0.2s
GaussianNB             roc_auc=0.4327 (+/-0.0268)  0.1s
LDA                    roc_auc=0.5015 (+/-0.0219)  0.1s
QDA                    FALHOU: LinAlgError: The covariance matrix of class 0 is not full rank. Increase the value of `reg_param` to reduce the collinearity.
DecisionTree           roc_auc=0.4829 (+/-0.0204)  1.8s
RandomForest           roc_auc=0.5175 (+/-0.0483)  4.0s
ExtraTrees             roc_auc=0.5179 (+/-0.0733)  1.9s
GradientBoosting       roc_auc=0.5110 (+/-0.0205)  0.3s
HistGradientBoosting   roc_auc=0.5130 (+/-0.0382)  0.4s
AdaBoost               roc_auc=0.5177 (+/-0.0612)  0.3s
MLP

KeyboardInterrupt: 

In [ ]:
# ==== 4.3 Leitura visual do torneio =======================================
fig, ax = plt.subplots(figsize=(9, 6))
top = ranking.head(15).iloc[::-1]
ax.barh(top["modelo"], top[f"{METRICA_ALVO}_mean"],
        xerr=top[f"{METRICA_ALVO}_std"], color="#4C72B0", alpha=.85)
ax.set_xlabel(METRICA_ALVO)
ax.set_title(f"Comparação de famílias — CV {CV.get_n_splits()} folds (só treino)")
ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

# Diagnóstico de overfitting: gap alto = modelo decorou o treino
ranking[["modelo", "train_alvo", f"{METRICA_ALVO}_mean", "gap_overfit", "fit_time_s"]].round(4)

In [ ]:
# ==== 4.4 Escolha da família vencedora ====================================
# Critério prático: melhor métrica dentro de 1 desvio-padrão -> desempata pelo
# modelo mais simples/rápido (navalha de Occam aplicada a ML).
melhor = ranking.iloc[0]
teto = melhor[f"{METRICA_ALVO}_mean"] - melhor[f"{METRICA_ALVO}_std"]
empatados = ranking[ranking[f"{METRICA_ALVO}_mean"] >= teto]
print("Estatisticamente empatados no topo:")
empatados[["modelo", f"{METRICA_ALVO}_mean", f"{METRICA_ALVO}_std", "fit_time_s"]].round(4)

MELHOR_FAMILIA = ranking.loc[0, "modelo"]
print(f"\nFamília vencedora: {MELHOR_FAMILIA}")

---
## 5. Escolha e Otimização (`RandomizedSearchCV`)

`RandomizedSearchCV` amostra `n_iter` combinações do espaço de busca. Com espaço grande
ele acha um ótimo tão bom quanto o `GridSearchCV` em uma fração do tempo — e aceita
distribuições contínuas (`loguniform`, `randint`), que a grade fixa não aceita.

Ordem de trabalho:
1. `RandomizedSearchCV` amplo (`n_iter` alto, faixas largas) → encontra a região boa;
2. opcionalmente um `GridSearchCV` estreito em volta do vencedor → refina.

Abaixo estão as grades das **todas as famílias**, prontas para copiar.
Prefixo `model__` porque o estimador é o passo `"model"` do Pipeline.

In [ ]:
# ==== 5.1 Grades de hiperparâmetros — repositório completo ================
from scipy.stats import loguniform, uniform, randint

GRIDS = {

# ------------------------------------------------------------------ LINEARES
"LogisticRegression": {
    "model__C":            loguniform(1e-4, 1e3),   # inverso da regularização (menor = mais regularizado)
    "model__penalty":      ["l1", "l2"],
    "model__solver":       ["liblinear", "saga"],   # únicos que aceitam L1
    "model__class_weight": [None, "balanced"],
    # elasticnet: solver="saga", penalty="elasticnet", model__l1_ratio: uniform(0, 1)
},

"RidgeClassifier": {
    "model__alpha":        loguniform(1e-3, 1e3),   # força da regularização (maior = mais)
    "model__class_weight": [None, "balanced"],
    "model__solver":       ["auto", "lsqr", "sparse_cg"],
},

"SGDClassifier": {
    "model__loss":          ["hinge", "log_loss", "modified_huber"],
    "model__penalty":       ["l1", "l2", "elasticnet"],
    "model__alpha":         loguniform(1e-6, 1e-1),
    "model__l1_ratio":      uniform(0, 1),
    "model__learning_rate": ["optimal", "adaptive"],
    "model__eta0":          loguniform(1e-4, 1e-1),  # só para "adaptive"
},

# ---------------------------------------------------------------------- SVM
"SVC_rbf": {
    "model__C":            loguniform(1e-2, 1e3),   # rigidez da margem
    "model__gamma":        loguniform(1e-5, 1e1),   # alcance do kernel; alto = superajuste
    "model__kernel":       ["rbf", "poly", "sigmoid"],
    "model__degree":       randint(2, 5),           # só para poly
    "model__class_weight": [None, "balanced"],
},

"LinearSVC": {
    "model__C":            loguniform(1e-3, 1e2),
    "model__loss":         ["hinge", "squared_hinge"],
    "model__class_weight": [None, "balanced"],
},

# ---------------------------------------------------------------- DISTÂNCIA
"KNN": {
    "model__n_neighbors": randint(1, 50),
    "model__weights":     ["uniform", "distance"],  # "distance" pondera vizinho próximo
    "model__metric":      ["euclidean", "manhattan", "minkowski", "chebyshev"],
    "model__p":           [1, 2],
    "model__algorithm":   ["auto", "ball_tree", "kd_tree"],
},

# ------------------------------------------------------------------ BAYESIANO
"GaussianNB": {
    "model__var_smoothing": loguniform(1e-12, 1e-2),  # estabilidade numérica
},
# MultinomialNB / ComplementNB (contagens, texto):
#   "model__alpha": loguniform(1e-3, 10), "model__fit_prior": [True, False]

# --------------------------------------------------------------- DISCRIMINANTE
"LDA": {
    "model__solver":   ["svd", "lsqr", "eigen"],
    "model__shrinkage": [None, "auto", 0.1, 0.5, 0.9],  # não vale com solver="svd"
},
"QDA": {
    "model__reg_param": uniform(0, 1),
},

# ------------------------------------------------------------------- ÁRVORE
"DecisionTree": {
    "model__criterion":         ["gini", "entropy", "log_loss"],
    "model__max_depth":         [None, 3, 5, 8, 12, 20],
    "model__min_samples_split": randint(2, 40),
    "model__min_samples_leaf":  randint(1, 30),
    "model__max_features":      [None, "sqrt", "log2"],
    "model__ccp_alpha":         loguniform(1e-5, 1e-1),  # poda por complexidade
    "model__class_weight":      [None, "balanced"],
},

# -------------------------------------------------------- ENSEMBLE: BAGGING
"RandomForest": {
    "model__n_estimators":      randint(200, 1200),   # mais é sempre >= melhor (só custa tempo)
    "model__max_depth":         [None, 5, 10, 20, 30],
    "model__min_samples_split": randint(2, 20),
    "model__min_samples_leaf":  randint(1, 15),
    "model__max_features":      ["sqrt", "log2", 0.3, 0.5, None],
    "model__bootstrap":         [True, False],
    "model__max_samples":       [None, 0.6, 0.8],     # só com bootstrap=True
    "model__class_weight":      [None, "balanced", "balanced_subsample"],
},

"ExtraTrees": {
    "model__n_estimators":      randint(200, 1200),
    "model__max_depth":         [None, 10, 20, 40],
    "model__min_samples_split": randint(2, 20),
    "model__min_samples_leaf":  randint(1, 15),
    "model__max_features":      ["sqrt", "log2", 0.5, None],
    "model__class_weight":      [None, "balanced"],
},

# ------------------------------------------------------- ENSEMBLE: BOOSTING
"GradientBoosting": {
    "model__n_estimators":      randint(100, 800),
    "model__learning_rate":     loguniform(1e-3, 0.3),  # troca com n_estimators
    "model__max_depth":         randint(2, 8),
    "model__min_samples_leaf":  randint(1, 30),
    "model__subsample":         uniform(0.6, 0.4),      # < 1 vira stochastic GB
    "model__max_features":      ["sqrt", "log2", None],
},

"HistGradientBoosting": {
    "model__learning_rate":       loguniform(1e-3, 0.3),
    "model__max_iter":            randint(100, 800),
    "model__max_leaf_nodes":      randint(10, 128),     # controla complexidade
    "model__max_depth":           [None, 4, 8, 16],
    "model__min_samples_leaf":    randint(5, 60),
    "model__l2_regularization":   loguniform(1e-6, 1e1),
    "model__max_bins":            [64, 128, 255],
},

"AdaBoost": {
    "model__n_estimators":  randint(50, 600),
    "model__learning_rate": loguniform(1e-3, 2.0),
    "model__estimator":     [DecisionTreeClassifier(max_depth=d) for d in (1, 2, 3, 5)],
},

# ------------------------------------------------------------- REDE NEURAL
"MLP": {
    "model__hidden_layer_sizes": [(50,), (100,), (200,), (100, 50), (128, 64, 32)],
    "model__activation":         ["relu", "tanh", "logistic"],
    "model__alpha":              loguniform(1e-6, 1e-1),   # regularização L2
    "model__learning_rate_init": loguniform(1e-4, 1e-1),
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, 256],
    "model__solver":             ["adam", "lbfgs"],
},

# --------------------------------------------------------- BOOSTING EXTERNO
"XGBoost": {
    "model__n_estimators":      randint(200, 1500),
    "model__learning_rate":     loguniform(1e-3, 0.3),
    "model__max_depth":         randint(2, 12),
    "model__min_child_weight":  randint(1, 20),        # peso mínimo por folha
    "model__subsample":         uniform(0.5, 0.5),     # amostragem de linhas
    "model__colsample_bytree":  uniform(0.4, 0.6),     # amostragem de colunas
    "model__gamma":             loguniform(1e-4, 5),   # ganho mínimo para dividir
    "model__reg_alpha":         loguniform(1e-4, 10),  # L1
    "model__reg_lambda":        loguniform(1e-3, 50),  # L2
    "model__scale_pos_weight":  [1, 3, 5, 10],         # desbalanceamento
},

"LightGBM": {
    "model__n_estimators":      randint(200, 1500),
    "model__learning_rate":     loguniform(1e-3, 0.3),
    "model__num_leaves":        randint(15, 255),      # principal controle do LGBM
    "model__max_depth":         [-1, 4, 8, 16],
    "model__min_child_samples": randint(5, 100),
    "model__subsample":         uniform(0.5, 0.5),
    "model__subsample_freq":    [0, 1, 5],             # subsample só age se freq > 0
    "model__colsample_bytree":  uniform(0.4, 0.6),
    "model__reg_alpha":         loguniform(1e-4, 10),
    "model__reg_lambda":        loguniform(1e-4, 10),
},

"CatBoost": {
    "model__iterations":     randint(200, 1500),
    "model__learning_rate":  loguniform(1e-3, 0.3),
    "model__depth":          randint(4, 10),
    "model__l2_leaf_reg":    loguniform(1e-1, 30),
    "model__border_count":   [32, 64, 128, 254],
    "model__random_strength": loguniform(1e-2, 10),
    "model__bagging_temperature": uniform(0, 1),
},
}

In [ ]:
# ==== 5.2 Grades dos blocos de pré-processamento ==========================
# Podem ser fundidas com a grade do modelo: o RandomizedSearch otimiza a esteira inteira.
GRID_PREP = {
    "prep__num__imputer__strategy": ["mean", "median", "most_frequent"],
    "prep__num__scaler":            [StandardScaler(), RobustScaler(), MinMaxScaler(),
                                     PowerTransformer(method="yeo-johnson")],
    "prep__cat__encoder__min_frequency": [0.005, 0.01, 0.05],
    # "pca__n_components": [0.90, 0.95, 0.99],
    # "select__k":         randint(5, 20),
    # "smote__sampling_strategy": uniform(0.3, 0.7),
}
# Uso: {**GRIDS["RandomForest"], **{"prep__num__imputer__strategy": [...]}}

In [ ]:
# ==== 5.3 A busca, no pipeline vencedor, só com dados de TREINO ===========
pipe_vencedor = PIPELINES[MELHOR_FAMILIA]
grid_vencedor = GRIDS[MELHOR_FAMILIA]

search = RandomizedSearchCV(
    estimator=pipe_vencedor,
    param_distributions=grid_vencedor,
    n_iter=60,                  # 50-100 costuma bastar; suba se o espaço for grande
    scoring=SCORING,
    refit=METRICA_ALVO,         # com scoring múltiplo, refit precisa nomear a métrica
    cv=CV,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=True,
    error_score=np.nan,         # combinação inválida não derruba a busca
)

search.fit(X_train, y_train)    # <- o teste continua lacrado

print("Melhor score CV:", round(search.best_score_, 4))
print("Melhores parâmetros:")
for k, v in sorted(search.best_params_.items()):
    print(f"  {k}: {v}")

best_model = search.best_estimator_   # já re-treinado em TODO o treino

In [ ]:
# ==== 5.4 Inspeção da busca ===============================================
cv_df = (pd.DataFrame(search.cv_results_)
           .sort_values(f"rank_test_{METRICA_ALVO}")
           .loc[:, lambda d: [c for c in d.columns
                              if c.startswith("param_")
                              or c in (f"mean_test_{METRICA_ALVO}",
                                       f"std_test_{METRICA_ALVO}",
                                       f"mean_train_{METRICA_ALVO}",
                                       "mean_fit_time")]])
display(cv_df.head(10).round(4))

# Sensibilidade: qual hiperparâmetro realmente move a métrica?
for col in [c for c in cv_df.columns if c.startswith("param_")][:4]:
    sub = cv_df[[col, f"mean_test_{METRICA_ALVO}"]].dropna()
    if sub[col].nunique() > 1:
        print(f"\n{col}")
        print(sub.groupby(col)[f"mean_test_{METRICA_ALVO}"].mean().sort_values(ascending=False).head())

In [ ]:
# ==== 5.5 Variantes de busca ==============================================

# --- Refino local com grade fixa em volta do vencedor --------------------
# grid_fino = {"model__max_depth": [8, 10, 12], "model__n_estimators": [500, 700, 900]}
# fino = GridSearchCV(pipe_vencedor, grid_fino, scoring=METRICA_ALVO,
#                     cv=CV, n_jobs=N_JOBS).fit(X_train, y_train)

# --- Halving: elimina candidatos ruins com poucos dados (bem mais rápido) --
# halving = HalvingRandomSearchCV(
#     pipe_vencedor, grid_vencedor, n_candidates=200, factor=3,
#     resource="n_samples", scoring=METRICA_ALVO, cv=CV,
#     n_jobs=N_JOBS, random_state=RANDOM_STATE).fit(X_train, y_train)

# --- Otimizar VÁRIAS famílias de uma vez (lista de espaços) ---------------
# from sklearn.base import BaseEstimator
# espaco = [
#     {"model": [LogisticRegression(max_iter=2000)], **GRIDS["LogisticRegression"]},
#     {"model": [RandomForestClassifier(random_state=RANDOM_STATE)], **GRIDS["RandomForest"]},
# ]
# RandomizedSearchCV(Pipeline([("prep", build_preprocessor("scaled")),
#                              ("model", LogisticRegression())]),
#                    espaco, n_iter=100, cv=CV)

# --- Busca bayesiana (Optuna): mais eficiente quando cada fit é caro ------
# import optuna
# def objective(trial):
#     params = {"model__max_depth": trial.suggest_int("model__max_depth", 3, 15),
#               "model__learning_rate": trial.suggest_float("model__learning_rate", 1e-3, .3, log=True)}
#     return cross_validate(pipe_vencedor.set_params(**params), X_train, y_train,
#                           cv=CV, scoring=METRICA_ALVO)["test_score"].mean()
# study = optuna.create_study(direction="maximize"); study.optimize(objective, n_trials=100)

---
## 6. A Prova Final (Inferência no Teste)

O teste é aberto **uma única vez**. Se você olhar o resultado, mexer no modelo e olhar de
novo, o teste virou validação e a estimativa deixou de ser honesta.

In [ ]:
# ==== 6.1 Predição ========================================================
y_pred = best_model.predict(X_test)

# Probabilidade da classe positiva (para ROC, PR e ajuste de threshold)
if hasattr(best_model, "predict_proba"):
    y_proba = best_model.predict_proba(X_test)[:, 1]
else:                                        # SVC(probability=False), RidgeClassifier...
    from scipy.special import expit
    y_proba = expit(best_model.decision_function(X_test))

In [ ]:
# ==== 6.2 Relatório de métricas ===========================================
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4))

metricas_finais = {
    "accuracy":          accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision":         precision_score(y_test, y_pred, zero_division=0),
    "recall":            recall_score(y_test, y_pred),
    "f1":                f1_score(y_test, y_pred),
    "f2 (recall pesa 2x)": fbeta_score(y_test, y_pred, beta=2),
    "roc_auc":           roc_auc_score(y_test, y_proba),
    "avg_precision":     average_precision_score(y_test, y_proba),
    "log_loss":          log_loss(y_test, y_proba),
    "brier":             brier_score_loss(y_test, y_proba),
    "mcc":               matthews_corrcoef(y_test, y_pred),
    "kappa":             cohen_kappa_score(y_test, y_pred),
}
display(pd.Series(metricas_finais).round(4).to_frame("teste"))

# Comparação honesta: CV de treino x teste. Queda grande = overfitting na busca.
print(f"CV (treino): {search.best_score_:.4f}  |  Teste: {metricas_finais[METRICA_ALVO]:.4f}")

In [ ]:
# ==== 6.3 Matriz de confusão ==============================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=CLASSES, cmap="Blues",
    colorbar=False, ax=axes[0])
axes[0].set_title("Contagens absolutas")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=CLASSES, cmap="Blues",
    normalize="true", values_format=".2%", colorbar=False, ax=axes[1])
axes[1].set_title("Normalizada por classe real (recall na diagonal)")

plt.tight_layout(); plt.show()

# Leitura numérica
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"VN={tn}  FP={fp} (alarme falso)  FN={fn} (deixou passar)  VP={tp}")
print(f"Especificidade (VN/(VN+FP)): {tn/(tn+fp):.4f}")
print(f"Sensibilidade  (VP/(VP+FN)): {tp/(tp+fn):.4f}")

In [ ]:
# ==== 6.4 Curvas ROC, Precisão-Revocação e Calibração =====================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0], name=MELHOR_FAMILIA)
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="acaso")
axes[0].set_title("ROC"); axes[0].legend()

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[1], name=MELHOR_FAMILIA)
axes[1].axhline(y_test.mean(), ls="--", c="k", lw=1, label="prevalência")
axes[1].set_title("Precisão x Revocação"); axes[1].legend()
# Com desbalanceamento forte, a curva PR é mais informativa que a ROC.

CalibrationDisplay.from_predictions(y_test, y_proba, n_bins=10, ax=axes[2], name=MELHOR_FAMILIA)
axes[2].set_title("Calibração (probabilidade prevista x observada)")

plt.tight_layout(); plt.show()

In [ ]:
# ==== 6.5 Ajuste do threshold =============================================
# 0.5 é apenas o padrão, não o ótimo. O corte certo depende do custo do erro.
prec, rec, thr = precision_recall_curve(y_test, y_proba)

# (a) Maximizar F1
f1s = 2 * prec * rec / (prec + rec + 1e-12)
thr_f1 = thr[np.nanargmax(f1s[:-1])]

# (b) Garantir um recall mínimo de negócio (ex.: capturar 80% dos churns)
RECALL_MIN = 0.80
viaveis = np.where(rec[:-1] >= RECALL_MIN)[0]
thr_recall = thr[viaveis[-1]] if len(viaveis) else 0.5

# (c) Minimizar custo esperado (o critério mais defensável)
CUSTO_FN, CUSTO_FP = 10.0, 1.0     # perder cliente custa 10x um contato à toa
grade = np.linspace(0.01, 0.99, 199)
custos = [(CUSTO_FN * ((y_proba < t) & (y_test == 1)).sum()
           + CUSTO_FP * ((y_proba >= t) & (y_test == 0)).sum()) for t in grade]
thr_custo = grade[int(np.argmin(custos))]

print(f"threshold F1 máximo:      {thr_f1:.3f}")
print(f"threshold recall >= {RECALL_MIN}: {thr_recall:.3f}")
print(f"threshold custo mínimo:   {thr_custo:.3f}  (custo={min(custos):.0f})")

THRESHOLD = thr_custo
y_pred_ajustado = (y_proba >= THRESHOLD).astype(int)
print(classification_report(y_test, y_pred_ajustado, target_names=CLASSES, digits=4))

# IMPORTANTE: escolher o threshold no TESTE já é uma forma leve de vazamento.
# Em produção, calibre o corte no conjunto de validação (etapa 2.2) ou via
# cross_val_predict sobre o treino; use o teste só para reportar.

In [ ]:
# ==== 6.6 Importância das variáveis =======================================
# Importância por permutação: funciona para QUALQUER modelo e mede impacto real
# na métrica (a importância nativa da árvore é enviesada para alta cardinalidade).
imp = permutation_importance(
    best_model, X_test, y_test,
    scoring=METRICA_ALVO, n_repeats=20,
    random_state=RANDOM_STATE, n_jobs=N_JOBS,
)
imp_df = (pd.DataFrame({"variavel": X_test.columns,
                        "importancia": imp.importances_mean,
                        "std": imp.importances_std})
            .sort_values("importancia", ascending=False))

fig, ax = plt.subplots(figsize=(8, 5))
top = imp_df.head(15).iloc[::-1]
ax.barh(top["variavel"], top["importancia"], xerr=top["std"], color="#55A868")
ax.set_title(f"Importância por permutação ({METRICA_ALVO})")
ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.show()

# Coeficientes (modelos lineares):
# nomes = best_model.named_steps["prep"].get_feature_names_out()
# coefs = pd.Series(best_model.named_steps["model"].coef_[0], index=nomes).sort_values()

# SHAP (explicação local + global, o padrão atual para boosting):
# import shap
# X_prep = best_model.named_steps["prep"].transform(X_test)
# ex = shap.TreeExplainer(best_model.named_steps["model"])
# shap.summary_plot(ex.shap_values(X_prep), X_prep)

In [ ]:
# ==== 6.7 Erros mais graves ===============================================
# Olhar os casos em que o modelo errou com mais confiança quase sempre revela
# um problema de dado ou uma variável faltando.
erros = X_test.copy()
erros["real"] = y_test.values
erros["prob"] = y_proba
erros["pred"] = y_pred_ajustado
erros["erro"] = erros["real"] != erros["pred"]

print("Falsos negativos mais confiantes (modelo disse 'não' e era 'sim'):")
display(erros[(erros.real == 1) & (erros.pred == 0)].nsmallest(5, "prob"))
print("Falsos positivos mais confiantes:")
display(erros[(erros.real == 0) & (erros.pred == 1)].nlargest(5, "prob"))

In [ ]:
# ==== 6.8 Modelo final e persistência =====================================
# Passo final: re-treinar nos dados COMPLETOS (treino + teste). O teste já
# cumpriu seu papel de estimar o desempenho; jogar fora 20% dos dados em
# produção não faz sentido.
from sklearn.base import clone
modelo_producao = clone(search.best_estimator_)   # copia a configuração, descarta o que já foi ajustado
modelo_producao.fit(X, y)

artefato = {
    "modelo": modelo_producao,
    "threshold": THRESHOLD,
    "classes": CLASSES,
    "features": {"num": NUM_FEATURES, "cat": CAT_FEATURES},
    "metricas_teste": metricas_finais,
    "params": search.best_params_,
    "sklearn_version": __import__("sklearn").__version__,
    "treinado_em": pd.Timestamp.now().isoformat(),
}
joblib.dump(artefato, "modelo_classificacao.joblib", compress=3)

# --- Carregar e usar ------------------------------------------------------
# art = joblib.load("modelo_classificacao.joblib")
# proba = art["modelo"].predict_proba(X_novo)[:, 1]
# pred  = (proba >= art["threshold"]).astype(int)

---
## Apêndice — Cheatsheet

### Qual família tentar primeiro
| Situação | Comece por |
|---|---|
| Tabular, qualquer tamanho | **HistGradientBoosting / LightGBM** (quase sempre o topo) |
| Precisa explicar a decisão | `LogisticRegression`, `DecisionTree` raso |
| Poucas linhas (< 1.000) | `LogisticRegression`, `SVC`, `LDA`, `GaussianNB` |
| Muitas colunas, poucas linhas (p >> n) | `LogisticRegression(penalty="l1")`, `LinearSVC` |
| Muitas linhas (> 1M) | `SGDClassifier`, `LightGBM`, `HistGradientBoosting` |
| Texto (TF-IDF) | `LinearSVC`, `ComplementNB`, `LogisticRegression` |
| Categóricas de alta cardinalidade | `CatBoost`, `HistGradientBoosting` |
| Fronteira muito não-linear, dado limpo | `SVC(kernel="rbf")` |
| Precisa de probabilidade confiável | Logística ou boosting + `CalibratedClassifierCV` |

### Sintomas e correções
| Sintoma | Causa provável | Correção |
|---|---|---|
| CV ótima, teste ruim | vazamento ou busca superajustada | pré-processo dentro do Pipeline; reduza `n_iter`; use CV repetida |
| Treino >> validação | overfitting | mais regularização, menos profundidade, mais dados |
| Treino ≈ validação, ambos ruins | underfitting | mais capacidade, mais variáveis, mais interações |
| Acurácia alta, recall zero | desbalanceamento | `class_weight`, mude a métrica, ajuste o threshold |
| Métrica instável entre folds | dataset pequeno | `RepeatedStratifiedKFold`, mais folds |
| Ótimo em CV, péssimo em produção | *drift* / grupo vazando | split por tempo ou por grupo |

### Multiclasse — o que muda
```python
# Métricas exigem "average": macro (todas as classes pesam igual) ou weighted (pesa por frequência)
f1_score(y_test, y_pred, average="macro")
roc_auc_score(y_test, y_proba_matriz, multi_class="ovr", average="macro")
SCORING = {"f1_macro": "f1_macro", "bal_acc": "balanced_accuracy"}

# Estratégias explícitas quando o modelo é binário por natureza:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
# Multirrótulo (várias classes por linha):
from sklearn.multioutput import MultiOutputClassifier
```

### Calibração de probabilidade
```python
# SVM e árvores produzem escores mal calibrados. Corrija sem tocar no teste:
calibrado = CalibratedClassifierCV(best_model, method="isotonic", cv=5)  # "sigmoid" se poucos dados
calibrado.fit(X_train, y_train)
```